# Modelo para el Analisis de factores que influyen en la popularidad de los videojuegos

Institucion: Escuela Politecnica Nacional EPN  
Materia: Metodos Numericos  
Semestre: 2026-A  
Estudiantes:  
* Ricardo Cisneros  
* Hugo Coyago  
* Jair Paez  

---

In [3]:
"""Importa las bibliotecas requeridas para el proyecto."""
import os
import csv
import re
import pandas as pd
import numpy as np

# Parte 1: Unificacion del Dataset
En esta etapa, reconstruimos el dataset a partir de los datos crudos extraidos de Steam y SteamSpy.

In [5]:
def analizar_rango_duenos(cadena_duenos: str) -> "int | str":
    """
    Calcula el punto medio del rango de duenios estimados.

    Args:
        cadena_duenos (str): Cadena de texto con el rango de duenios (ej. "10,000 .. 20,000").
        
    Returns:
        "int | str": El punto medio del rango como valor entero, o una cadena vacia en caso de error.
    """
    if not cadena_duenos or '..' not in cadena_duenos:
        return ""
    partes = cadena_duenos.split('..')
    if len(partes) == 2:
        try:
            limite_inferior = int(partes[0].replace(',', '').strip())
            limite_superior = int(partes[1].replace(',', '').strip())
            return (limite_inferior + limite_superior) // 2
        except ValueError:
            return ""
    return ""

def extraer_ano_lanzamiento(fecha_str: str) -> str:
    """
    Extrae el ano de lanzamiento desde una fecha dada en formato texto.

    Args:
        fecha_str (str): Cadena de texto que contiene la fecha de lanzamiento.

    Returns:
        str: El ano extraido (ej. "2015"), o una cadena vacia si no se encuentra coincidencia.
    """
    if not fecha_str or fecha_str == "\\N":
        return ""
    coincidencia = re.search(r'\b(19\d\d|20\d\d)\b', fecha_str)
    if coincidencia:
        return coincidencia.group(1)
    return ""

def unificar_dataset() -> str:
    """
    Ejecuta la unificacion de todos los datos en un solo conjunto de datos consolidados.

    Returns:
        str: La ruta del archivo CSV donde se guardaron los datos unificados.
    """
    print("Iniciando unificacion de datos...")

    directorio_base = ".."
    juegos_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "games", "games.csv")
    resenas_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "reviews", "reviews.csv")
    steamspy_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "steamspy_insights", "steamspy_insights.csv")
    categorias_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "categories", "categories.csv")
    salida_datos_unificados_crudos = os.path.join(directorio_base, "01_Datos", "steam_raw_unified_all_columns.csv")

    datos_categorias = {}
    with open(categorias_csv, mode='r', encoding='utf-8') as archivo_cat:
        lector = csv.DictReader(archivo_cat)
        for fila in lector:
            id_app = fila.get("app_id")
            if id_app:
                datos_categorias[id_app] = datos_categorias.get(id_app, 0) + 1

    datos_juegos = {}
    with open(juegos_csv, mode='r', encoding='utf-8') as archivo_juegos:
        lector = csv.DictReader(archivo_juegos)
        for fila in lector:
            id_aplicacion = fila.get("app_id")
            if not id_aplicacion:
                continue
            
            idiomas_str = fila.get("languages", "").strip()
            cant_idiomas = 0
            if idiomas_str and idiomas_str != "\\N":
                cant_idiomas = len([lang.strip() for lang in idiomas_str.split(",") if lang.strip()])

            es_gratis_str = fila.get("is_free", "").strip().lower()
            es_gratis_num = 1 if es_gratis_str == "true" else (0 if es_gratis_str == "false" else "")

            datos_juegos[id_aplicacion] = {
                "name": fila.get("name", ""),
                "release_date": fila.get("release_date", ""),
                "release_year": extraer_ano_lanzamiento(fila.get("release_date", "")),
                "is_free": es_gratis_num,
                "price_overview": fila.get("price_overview", ""),
                "games_languages": fila.get("languages", ""),
                "language_count": cant_idiomas,
                "type": fila.get("type", "")
            }

    datos_resenas = {}
    with open(resenas_csv, mode='r', encoding='utf-8') as archivo_resenas:
        lector = csv.DictReader(archivo_resenas)
        for fila in lector:
            id_aplicacion = fila.get("app_id")
            if not id_aplicacion:
                continue
            datos_resenas[id_aplicacion] = {
                "review_score": fila.get("review_score", ""),
                "review_score_description": fila.get("review_score_description", ""),
                "reviews_positive": fila.get("positive", ""),
                "reviews_negative": fila.get("negative", ""),
                "reviews_total": fila.get("total", ""),
                "metacritic_score": fila.get("metacritic_score", ""),
                "reviews_text": fila.get("reviews", ""),
                "recommendations": fila.get("recommendations", ""),
                "steamspy_user_score": fila.get("steamspy_user_score", ""),
                "steamspy_score_rank": fila.get("steamspy_score_rank", ""),
                "reviews_steamspy_positive": fila.get("steamspy_positive", ""),
                "reviews_steamspy_negative": fila.get("steamspy_negative", "")
            }

    datos_steamspy = {}
    todos_los_ids_aplicacion = set(datos_juegos.keys()) | set(datos_resenas.keys())
    with open(steamspy_csv, mode='r', encoding='utf-8') as archivo_steamspy:
        lector = csv.DictReader(archivo_steamspy)
        for fila in lector:
            id_aplicacion = fila.get("app_id")
            if not id_aplicacion:
                continue
            datos_steamspy[id_aplicacion] = fila
            todos_los_ids_aplicacion.add(id_aplicacion)

    todos_los_datos_fusionados = []
    for id_aplicacion in sorted(list(todos_los_ids_aplicacion), key=lambda x: int(x) if x.isdigit() else 9999999):
        juego = datos_juegos.get(id_aplicacion, {
            "name": "", "release_date": "", "release_year": "", "is_free": "", "price_overview": "",
            "games_languages": "", "language_count": "", "type": ""
        })
        resena = datos_resenas.get(id_aplicacion, {
            "review_score": "", "review_score_description": "", "reviews_positive": "", "reviews_negative": "",
            "reviews_total": "", "metacritic_score": "", "reviews_text": "", "recommendations": "",
            "steamspy_user_score": "", "steamspy_score_rank": "", "reviews_steamspy_positive": "", "reviews_steamspy_negative": ""
        })
        spy = datos_steamspy.get(id_aplicacion, {
            "developer": "", "publisher": "", "owners_range": "", "concurrent_users_yesterday": "",
            "playtime_average_forever": "", "playtime_average_2weeks": "", "playtime_median_forever": "",
            "playtime_median_2weeks": "", "price": "", "initial_price": "", "discount": "", "languages": "", "genres": ""
        })

        precio_dolares = ""
        if spy.get("price", "") and spy.get("price", "") != "\\N":
            try:
                precio_dolares = round(int(spy.get("price", "")) / 100.0, 2)
            except ValueError:
                pass

        precio_inicial_dolares = ""
        if spy.get("initial_price", "") and spy.get("initial_price", "") != "\\N":
            try:
                precio_inicial_dolares = round(int(spy.get("initial_price", "")) / 100.0, 2)
            except ValueError:
                pass

        proporcion_aprobacion = ""
        if resena.get("reviews_positive", "") and resena.get("reviews_negative", ""):
            try:
                positivas = int(resena.get("reviews_positive", ""))
                negativas = int(resena.get("reviews_negative", ""))
                if positivas + negativas > 0:
                    proporcion_aprobacion = round(positivas / (positivas + negativas), 4)
            except ValueError:
                pass

        fila_fusionada = {
            "app_id": id_aplicacion,
            "name": juego["name"],
            "release_date": juego["release_date"],
            "release_year": juego["release_year"],
            "is_free": juego["is_free"],
            "price_overview": juego["price_overview"],
            "games_languages": juego["games_languages"],
            "language_count": juego["language_count"],
            "category_count": datos_categorias.get(id_aplicacion, 0),
            "type": juego["type"],
            "developer": spy.get("developer", ""),
            "publisher": spy.get("publisher", ""),
            "owners_range": spy.get("owners_range", ""),
            "concurrent_users_yesterday": spy.get("concurrent_users_yesterday", ""),
            "playtime_average_forever": spy.get("playtime_average_forever", ""),
            "playtime_average_2weeks": spy.get("playtime_average_2weeks", ""),
            "playtime_median_forever": spy.get("playtime_median_forever", ""),
            "playtime_median_2weeks": spy.get("playtime_median_2weeks", ""),
            "steamspy_price": spy.get("price", ""),
            "steamspy_initial_price": spy.get("initial_price", ""),
            "steamspy_discount": spy.get("discount", ""),
            "steamspy_languages": spy.get("languages", ""),
            "genres": spy.get("genres", ""),
            "review_score": resena["review_score"],
            "review_score_description": resena["review_score_description"],
            "reviews_positive": resena["reviews_positive"],
            "reviews_negative": resena["reviews_negative"],
            "reviews_total": resena["reviews_total"],
            "metacritic_score": resena["metacritic_score"],
            "reviews_text": resena["reviews_text"],
            "recommendations": resena["recommendations"],
            "steamspy_user_score": resena["steamspy_user_score"],
            "steamspy_score_rank": resena["steamspy_score_rank"],
            "reviews_steamspy_positive": resena["reviews_steamspy_positive"],
            "reviews_steamspy_negative": resena["reviews_steamspy_negative"],
            "price_usd": precio_dolares,
            "steamspy_initial_price_usd": precio_inicial_dolares,
            "approval_ratio": proporcion_aprobacion,
            "estimated_owners": analizar_rango_duenos(spy.get("owners_range", ""))
        }
        todos_los_datos_fusionados.append(fila_fusionada)

    encabezados = list(todos_los_datos_fusionados[0].keys())
    with open(salida_datos_unificados_crudos, mode='w', newline='', encoding='utf-8') as archivo_salida:
        escritor = csv.DictWriter(archivo_salida, fieldnames=encabezados)
        escritor.writeheader()
        escritor.writerows(todos_los_datos_fusionados)

    print("Consolidacion completada con exito.")
    return salida_datos_unificados_crudos

# Ejecucion de la funcion principal de la Parte 1
archivo_unificado = unificar_dataset()


Iniciando unificacion de datos...
Consolidacion completada con exito.


# Parte 2: Limpieza de Datos

Proceso de limpieza y tratamiento de nulos.

In [ ]:
def limpiar_datos(ruta_archivo: str) -> "tuple[pd.DataFrame, list[str]]":
    """
    Limpia los datos, procesa nulos y filtra variables independientes de alta calidad.

    Args:
        ruta_archivo (str): Ruta al archivo CSV con los datos crudos unificados.

    Returns:
        "tuple[pd.DataFrame, list[str]]": Un DataFrame limpio y la lista de variables clave.
    """
    df = pd.read_csv(ruta_archivo, dtype={'app_id': str}, low_memory=False)

    """Define las variables clave del modelo."""
    variable_objetivo = 'estimated_owners'
    variables_predictoras = [
        'price_usd', 'is_free', 'approval_ratio', 'reviews_total',
        'language_count', 'category_count', 'release_year', 'genres'
    ]
    todas_las_variables = [variable_objetivo] + variables_predictoras

    """Limpia los registros duplicados."""
    df = df.drop_duplicates(subset=['app_id'])

    """Convierte los valores nulos a NaN y elimina las filas incompletas."""
    for col in todas_las_variables:
        if col != 'genres':
            df[col] = df[col].replace([r'\\\\N', 'N/A', 'nan', 'NaN', ''], np.nan)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df_limpio = df.dropna(subset=todas_las_variables).copy()

    """Filtra los valores incoherentes (precios negativos, variables fuera de rango)."""
    df_limpio = df_limpio[
        (df_limpio['price_usd'] >= 0) &
        (df_limpio['is_free'].isin([0.0, 1.0, 0, 1])) &
        (df_limpio['reviews_total'] >= 0) &
        (df_limpio['language_count'] >= 0) &
        (df_limpio['category_count'] >= 0) &
        (df_limpio['release_year'] >= 1997) & (df_limpio['release_year'] <= 2024) &
        (df_limpio['approval_ratio'] >= 0.0) & (df_limpio['approval_ratio'] <= 1.0) &
        (df_limpio['estimated_owners'] > 0)
    ]
    
    print("Limpieza de datos completada.")
    return df_limpio, todas_las_variables

# Ejecucion de la funcion principal de la Parte 2
df_limpio, todas_las_variables = limpiar_datos(archivo_unificado)


# Parte 3: Normalizacion de Datos

Escalamiento de variables usando Z score .

In [ ]:
def normalizar_datos(df_limpio: pd.DataFrame, todas_las_variables: "list[str]") -> "tuple[pd.DataFrame, list[str]]":
    """
    Normaliza los datos numéricos y exporta el conjunto final a un CSV.

    Args:
        df_limpio (pd.DataFrame): DataFrame con los datos previamente limpiados.
        todas_las_variables ("list[str]"): Lista de las variables clave del dataset.

    Returns:
        "tuple[pd.DataFrame, list[str]]": El DataFrame con variables normalizadas y la lista de variables numéricas procesadas.
    """
    """Aplica la transformación logarítmica a reviews_total para reducir el sesgo extremo y alinearse con Grm y Šubelj (2021)."""
    if 'reviews_total' in df_limpio.columns:
        df_limpio['reviews_total'] = np.log10(df_limpio['reviews_total'] + 1)

    """Normaliza las variables usando el metodo Z-score (estandarizacion con media y desviacion estandar)."""
    variables_numericas = [col for col in todas_las_variables if col != 'genres']
    for col in variables_numericas:
        mean_val = df_limpio[col].mean()
        std_val = df_limpio[col].std()
        df_limpio[col + '_normalized'] = (df_limpio[col] - mean_val) / std_val if std_val > 0.0 else 0.0

    """Exporta el dataset limpio a un archivo CSV."""
    directorio_base = ".."
    archivo_salida = os.path.join(directorio_base, "01_Datos", "steam_cleaned_data.csv")
    df_limpio.to_csv(archivo_salida, index=False)
    
    print("Normalización completada y exportada con éxito.")
    return df_limpio, variables_numericas

# Ejecución de la función principal de la Parte 3
df_final, variables_numericas = normalizar_datos(df_limpio, todas_las_variables)
df_final[[c + '_normalized' for c in variables_numericas] + ['genres']].head()